In [2]:
####################
# Step 1: Define your model(s) (Dynamic Section)
####################

def laccurve5_dijkstra(self, stateVars, t, prev_var_return=None):
    # prev_var_return is required if you want to have a variable with an initial value that then gets updated each iteration
    # The variable must also be included in the outputs
    # TODO ask John about this issue, how common is it to have variables like this

    import numpy as np
    import math

    # Assign Parameter Values #
    a = self.parameters['a']
    b = self.parameters['b']
    b0 = self.parameters['b0']
    c = self.parameters['c']
    L = self.parameters['L']
    KDMIMILK = self.parameters['KDMIMILK']
    KDMImbw = self.parameters['KDMImbw']
    Hcfeed = self.parameters['Hcfeed']
    Hcmilk = self.parameters['Hcmilk']
    KL = self.parameters['KL']
    expL = self.parameters['expL']
    pregdate = self.parameters['pregdate']
    kf1 = self.parameters['kf1']
    kf2 = self.parameters['kf2']
    milk_value = self.parameters['milk_value']

    # Paramaters that require an initial value #
    if t == 0:
            E = self.parameters['E']
    else:
        E = prev_var_return[5]


    # Variables w/ Differential Equation #
    BW = stateVars[0]
    ER = stateVars[1]
    milkcumul = stateVars[2]
    DMIcumul = stateVars[3]
    IOFCcumul = stateVars[4]

    # Model Equations # 
    feed_price = (101 * Hcfeed + 2.7) / 1000

    BFat = ER/9.0       
    # Body fat

    Lmod = 1.0-(1.0-L)/(1.0+(KL/BFat)**expL)         
        # 1 - (1-L) = L?
        # Modifying value of L? Why the adjustment?

    dijkstra_milk = a * np.exp(b * (1 - np.exp(-b0 * t)) / b0 - c * t)
    # John has modified S to give 4% FCM yield per alveolus 

    NEmilk = E**Lmod*dijkstra_milk                         
        # Net energy milk
        # On slide 28 this is equation for FCM yield but L is modified here
        # Is L modified to give a NEmilk as well as a milk yield (milk)

    milk = NEmilk/Hcmilk                
        # Milk production

    DMINRC = (KDMIMILK * milk + KDMImbw * BW**0.75)*(1.0-math.exp(-0.192*(t/7+3.67)))   
        # Slide 28
        # NRC DMI equation

    fdbk = (kf1*t/7+kf2)*(ER-self.iStateVars[1])/self.iStateVars[1]       
    # Ellis 2006
        # Feedback on DMI

    DMI = DMINRC - fdbk                             
        # Dry matter intake     

    NEI = Hcfeed*DMI                                
        # Net energy intake

    NEmaint = 0.096*BW**0.75                        
        # Net energy maintenance

    if t < pregdate + 190:                          
        NEgest = 0
    else:
        NEgest = (0.00318*(t-pregdate-190)-0.0352)/0.218
        # Net energy gestation

    NEreqt = NEmaint + dijkstra_milk + NEgest + 5.12*2.0          
        # NE requirement, from NRC

    E = NEI/NEreqt            
        # Energy balance

    # Differential Equations # 
    dERdt = NEI - NEmaint - NEmilk - NEgest
    # Energy requirement (NE)

    if dERdt < 0:
        dBWdt = dERdt/4.92  # 4.92 Mcal/kg in NRC(1988)
    else:
        dBWdt = dERdt/5.12  # 5.12 Mcal/kg for gain
        # Bodyweight gain

    IOFC = (milk * milk_value) - (DMI * feed_price)

    # Format data for return # 
    differential_return = [dBWdt, dERdt, milk, DMI, IOFC]
    local_variables = locals()
    # Store local variables 
    variable_returns = [local_variables.get(variable_name) for variable_name in self.outputs]
    # Create list of variables to return

    return differential_return, variable_returns

functions_to_include = [laccurve5_dijkstra]



In [3]:
####################
# Step 2: Generate cows from dataset
####################

import pandas as pd
import simoolator as moo

# Import cow data 
cow_data_raw = pd.read_csv('../data/cow_import_data.csv')

# Function to convert a .csv to the required format
def cow_wrangler(df):
    # Initialize empty lists to store data for each cow
    cow_ids = []
    iStateVars_list = []
    parameters_dict_list = []

    # Iterate over rows
    for index, row in df.iterrows():
        # Extract cow ID
        cow_id = row['cow_ID']
        cow_ids.append(cow_id)

        # Extract iStateVars
        iStateVars = [row[col] for col in df.columns if col.startswith('iStateVar_')]
        iStateVars_list.append(iStateVars)

        # Extract parameters
        parameters_dict = {col.replace('param_', ''): row[col] for col in df.columns if col.startswith('param_')}
        parameters_dict_list.append(parameters_dict)

    # Create a new DataFrame
    wrangled_data = pd.DataFrame({
        'cow_ID': cow_ids,
        'iStateVars': iStateVars_list,
        'parameters': parameters_dict_list
    })

    return wrangled_data

cow_data_wranlged = cow_wrangler(cow_data_raw)

# Initalize herd
output = ['t', 'dijkstra_milk', 'NEmilk', 'milk', 'DMI', 'E', 'BW', 'BFat', 'ER', 'IOFC', 'milkcumul', 'DMIcumul', 'IOFCcumul'] # Model outputs
herd = moo.Herd(outputs=output, model_functions=functions_to_include)

# Load cows from dataframe
herd.import_dataframe(df=cow_data_wranlged)


All cows have been added
